# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

### [Bài 10] Trả lời Task 1: các thành phần của bài toán tìm kiếm

Bàn có `rows × cols` **chấm** (mặc định của đề là 4×4 chấm, tức 3×3 ô). Hai người chơi là **+1** (đi trước) và **−1**.

| Thành phần | Định nghĩa |
| :--- | :--- |
| **Trạng thái ban đầu** | Bàn chưa có đoạn nào (`lines` rỗng), chưa ô nào có chủ (`boxes` rỗng), người +1 đến lượt. |
| **Hành động** `actions(s)` | Vẽ một đoạn ngang `('h', r, c)` hoặc dọc `('v', r, c)` chưa vẽ, nối hai chấm kề nhau. Bàn `rows × cols` chấm có $L = rows\cdot(cols-1) + (rows-1)\cdot cols$ đoạn (4×4 chấm: $L = 24$). |
| **Mô hình chuyển trạng thái** `result(s, a)` | Thêm đoạn `a` vào `lines`. Mỗi ô kề `a` (tối đa 2 ô) vừa đủ 4 cạnh thì ghi `(hàng, cột): người chơi` vào `boxes`. Nếu nước đi **đóng được ít nhất một ô thì người đó đi tiếp**, ngược lại đổi lượt. |
| **Kiểm tra kết thúc** `terminal(s)` | Đã vẽ đủ cả $L$ đoạn (lúc đó mọi ô đều đã có chủ). |
| **Utility** `utility(s, p)` | So số ô của hai người theo góc nhìn người `p`: +1 nếu nhiều ô hơn, −1 nếu ít hơn, 0 nếu bằng nhau (chỉ xảy ra khi tổng số ô chẵn). |

Khác với Tic-Tac-Toe, lượt đi **không luân phiên cố định** vì luật đi tiếp, nên trạng thái đầy đủ dùng cho tìm kiếm là
cặp *(bàn cờ, người đến lượt)*. Bàn cờ giữ đúng cấu trúc dict chung của đề (`size`, `lines`, `boxes`) để các agent
đấu được với nhau; người đến lượt do môi trường quản lý (hàm `next_player` ở Task 2).

How big is the state space? Give an estimate and explain it.

### [Bài 10] Trả lời: kích thước không gian trạng thái

Một trạng thái gồm: tập đoạn đã vẽ, chủ của từng ô đã đóng và người đến lượt.

- **Cận dưới $2^L$**: mỗi đoạn vẽ hoặc chưa vẽ, và mọi tập con của $L$ đoạn đều đạt được (vẽ theo thứ tự bất kỳ),
  nên có ít nhất $2^L$ trạng thái khác nhau.
- **Cận trên $2^L \cdot 2^B \cdot 2$** với $B$ là số ô: mỗi ô đã đóng có tối đa 2 khả năng về chủ, cộng 2 khả năng người đến lượt.
- **Bàn 4×4 chấm** ($L = 24$, $B = 9$): $2^{24} \approx 1{,}7 \cdot 10^{7} \le |S| \le 2^{34} \approx 1{,}7 \cdot 10^{10}$.

Bảng dưới tính các con số này cho nhiều kích thước: mỗi hàng hoặc cột chấm thêm vào làm $L$ tăng, nên số trạng thái tăng
theo hàm mũ (bàn chuẩn 5×5 chấm đã có ít nhất $2^{40} \approx 10^{12}$ trạng thái). Để ra quyết định, vị trí ô của ai
không quan trọng, chỉ cần **hiệu số ô** hiện tại, nên số trạng thái cần phân biệt thực tế nhỏ hơn cận trên nhiều.

In [1]:
# [Bài 10] Ước lượng kích thước không gian trạng thái và cây trò chơi theo kích thước bàn (số chấm).
import math
import pandas as pd

def count_lines(rows, cols):
    """Số đoạn thẳng có thể vẽ trên bàn rows × cols chấm."""
    return rows * (cols - 1) + (rows - 1) * cols

def count_boxes(rows, cols):
    return (rows - 1) * (cols - 1)

size_rows = []
for rows, cols in [(2, 2), (2, 3), (3, 3), (3, 4), (4, 4), (5, 5), (6, 6)]:
    L, B = count_lines(rows, cols), count_boxes(rows, cols)
    size_rows.append({
        'bàn (chấm)': f'{rows}×{cols}', 'số ô': B, 'số đoạn L': L,
        'tập đoạn đã vẽ 2^L': f'{2 ** L:.3g}',
        'cận trên trạng thái 2^L·2^B·2': f'{2 ** (L + B + 1):.3g}',
        'số ván (lá) L!': f'{math.factorial(L):.3g}',
        'số nút cây ≈ e·L!': f'{sum(math.perm(L, k) for k in range(L + 1)):.3g}',
    })
size_table = pd.DataFrame(size_rows)
size_table

,bàn (chấm),số ô,số đoạn L,tập đoạn đã vẽ 2^L,cận trên trạng thái 2^L·2^B·2,số ván (lá) L!,số nút cây ≈ e·L!
0,2×2,1,4,16,64,24,65
1,2×3,2,7,128,1.02e+03,5.04e+03,1.37e+04
2,3×3,4,12,4.1e+03,1.31e+05,4.79e+08,1.3e+09
3,3×4,6,17,1.31e+05,1.68e+07,3.56e+14,9.67e+14
4,4×4,9,24,1.68e+07,1.72e+10,6.2e+23,1.69e+24
5,5×5,16,40,1.1e+12,1.44e+17,8.16e+47,2.22e+48
6,6×6,25,60,1.15e+18,7.74e+25,8.32e+81,2.26e+82


How big is the game tree that minimax search will go through? Give an estimate and explain it.

### [Bài 10] Trả lời: kích thước cây trò chơi mà Minimax phải duyệt

- Mỗi ván **luôn kéo dài đúng $L$ nước**: mỗi nước vẽ một đoạn; luật đi tiếp chỉ đổi *ai* đi, không đổi *số* nước.
- Mọi thứ tự vẽ $L$ đoạn đều là một ván hợp lệ, nên cây Minimax đầy đủ (không cắt tỉa) có **$L!$ lá**, và tổng số nút là
  $\sum_{k=0}^{L} \frac{L!}{(L-k)!} \approx e \cdot L!$ (hai cột cuối của bảng trên).
- Bàn 4×4 chấm: $24! \approx 6{,}2 \cdot 10^{23}$ lá, khoảng $1{,}7 \cdot 10^{24}$ nút. Ngay cả bàn 3×3 chấm (chỉ 4 ô) đã có
  $12! \approx 4{,}8 \cdot 10^{8}$ lá; bàn chuẩn 5×5 chấm khoảng $10^{48}$ nút.
- Cây lớn hơn không gian trạng thái rất nhiều vì **cùng một trạng thái đạt được qua nhiều thứ tự vẽ khác nhau**
  (ví dụ bàn 4×4: $\approx 10^{24}$ nút cây so với tối đa $\approx 10^{10}$ trạng thái). Vì vậy Minimax/Alpha-Beta đầy đủ
  chỉ khả thi trên bàn rất nhỏ; với bàn lớn hơn cần cắt tỉa, sắp xếp nước đi, heuristic và cutoff (Task 3, Task 4).

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [2]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

**[Bài 10] Nhận xét ô gốc của đề.** Ô trên minh họa cấu trúc dict chung (`size`, `lines`, `boxes`); nhóm giữ đúng cấu trúc này.
Có hai chỗ trong ô mẫu nhóm không dùng lại nguyên văn:
- `'boxes': dict` gán **kiểu** `dict` chứ không phải một dict rỗng, nên hàm `new_board` bên dưới tạo `'boxes': dict()`.
- `draw_line` kiểm tra biên chưa khớp với quy ước "`size` là số hàng, số cột **chấm**": trên bàn 4×4 chấm nó vẫn nhận
  `('h', 3, 3)` (chấm `(3, 4)` không tồn tại) và `('v', 3, 0)` (chấm `(4, 0)` không tồn tại). Hàm `is_valid_line` ở Task 2
  kiểm tra đúng: đoạn ngang cần $0 \le c < cols-1$, đoạn dọc cần $0 \le r < rows-1$.

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

**[Bài 10] Cách hiển thị.** `+` là chấm, `---` là đoạn ngang, `|` là đoạn dọc đã vẽ; chữ trong ô là người đã đóng ô
(**A** = người +1, **B** = người −1). Dòng cuối in số ô mỗi người và số đoạn đã vẽ. Ví dụ dưới là bàn 3×4 chấm có 6 đoạn,
người +1 đã đóng ô (0, 0).

In [3]:
# [Bài 10] Biểu diễn bàn cờ (dict như đề) và hàm hiển thị dạng chữ.
# Quy ước: 'size' = (số hàng chấm, số cột chấm); đoạn ('h', r, c) nối chấm (r, c)–(r, c+1);
# đoạn ('v', r, c) nối chấm (r, c)–(r+1, c); ô (r, c) có góc trên-trái là chấm (r, c).
# Người chơi là +1 và -1; khi hiển thị, ô của +1 ghi 'A', ô của -1 ghi 'B'.
import numpy as np

def new_board(rows=4, cols=4):
    """Bàn trống rows × cols chấm, cùng cấu trúc dict với đề."""
    return {'size': (rows, cols), 'lines': dict(), 'boxes': dict()}

def show_board(board):
    """In bàn cờ: '+' là chấm, '---' và '|' là đoạn đã vẽ, chữ trong ô là người đã đóng ô."""
    rows, cols = board['size']
    mark = {1: 'A', -1: 'B'}
    out = []
    for r in range(rows):
        dots = ''
        for c in range(cols):
            dots += '+'
            if c < cols - 1:
                dots += '---' if ('h', r, c) in board['lines'] else '   '
        out.append(dots)
        if r < rows - 1:
            sides = ''
            for c in range(cols):
                sides += '|' if ('v', r, c) in board['lines'] else ' '
                if c < cols - 1:
                    sides += f" {mark[board['boxes'][(r, c)]]} " if (r, c) in board['boxes'] else '   '
            out.append(sides)
    print('\n'.join(out))
    owners = list(board['boxes'].values())
    print(f"A (+1): {owners.count(1)} ô   B (-1): {owners.count(-1)} ô   "
          f"đã vẽ {len(board['lines'])} đoạn")

demo = new_board(3, 4)
for line in [('h', 0, 0), ('h', 1, 0), ('v', 0, 0), ('v', 0, 1), ('h', 0, 2), ('v', 1, 3)]:
    demo['lines'][line] = True
demo['boxes'][(0, 0)] = 1
show_board(demo)

+---+   +---+
| A |        
+---+   +   +
            |
+   +   +   +
A (+1): 1 ô   B (-1): 0 ô   đã vẽ 6 đoạn


Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

In [4]:
# [Bài 10] actions, result, terminal, utility (chạy được với mọi kích thước bàn).

def is_valid_line(board, line):
    rows, cols = board['size']
    orientation, r, c = line
    if orientation == 'h':
        return 0 <= r < rows and 0 <= c < cols - 1
    if orientation == 'v':
        return 0 <= r < rows - 1 and 0 <= c < cols
    return False

def all_lines(board):
    rows, cols = board['size']
    return ([('h', r, c) for r in range(rows) for c in range(cols - 1)] +
            [('v', r, c) for r in range(rows - 1) for c in range(cols)])

def box_sides(box):
    r, c = box
    return [('h', r, c), ('h', r + 1, c), ('v', r, c), ('v', r, c + 1)]

def adjacent_boxes(board, line):
    """Các ô (tối đa 2) có cạnh là đoạn `line`."""
    rows, cols = board['size']
    orientation, r, c = line
    candidates = [(r - 1, c), (r, c)] if orientation == 'h' else [(r, c - 1), (r, c)]
    return [(br, bc) for br, bc in candidates if 0 <= br < rows - 1 and 0 <= bc < cols - 1]

def actions(board):
    """Các đoạn còn vẽ được."""
    return [line for line in all_lines(board) if line not in board['lines']]

def result(board, player, action):
    """Bàn mới sau khi `player` (+1/-1) vẽ đoạn `action`; không sửa bàn cũ.
    Ô nào vừa đủ 4 cạnh thì ghi (row, col): player vào board['boxes']."""
    if not is_valid_line(board, action) or action in board['lines']:
        raise ValueError(f'Nước đi không hợp lệ {action} của người chơi {player}')
    new = {'size': board['size'], 'lines': dict(board['lines']), 'boxes': dict(board['boxes'])}
    new['lines'][action] = True
    for box in adjacent_boxes(new, action):
        if all(side in new['lines'] for side in box_sides(box)):
            new['boxes'][box] = player
    return new

def closed_box(board, new_board):
    """Nước vừa đi có đóng được ô nào không (đóng ô thì được đi tiếp)."""
    return len(new_board['boxes']) > len(board['boxes'])

def next_player(board, new_board, player):
    """Người đi tiếp: vẫn là `player` nếu vừa đóng ô, ngược lại đổi lượt."""
    return player if closed_box(board, new_board) else -player

def terminal(board):
    """Hết đoạn để vẽ thì ván kết thúc."""
    rows, cols = board['size']
    return len(board['lines']) == rows * (cols - 1) + (rows - 1) * cols

def score(board, player=1):
    """Số ô của `player` trừ số ô của đối thủ."""
    owners = list(board['boxes'].values())
    return owners.count(player) - owners.count(-player)

def utility(board, player=1):
    """+1 thắng, -1 thua, 0 hòa theo góc nhìn `player`; None nếu ván chưa kết thúc."""
    if not terminal(board):
        return None
    return int(np.sign(score(board, player)))

In [5]:
# [Bài 10] Kiểm tra các hàm trên những tình huống cần thiết.
b = new_board(3, 3)                       # 3×3 chấm = 4 ô, 12 đoạn
print('Bàn trống:', len(actions(b)), 'đoạn vẽ được; kết thúc?', terminal(b))
assert len(actions(b)) == 12 and not terminal(b) and utility(b) is None

# 1) Vẽ 3 cạnh của ô (0, 0), người -1 vẽ cạnh thứ 4 thì đóng ô và được đi tiếp.
for line in [('h', 0, 0), ('v', 0, 0), ('h', 1, 0)]:
    b = result(b, 1, line)
after = result(b, -1, ('v', 0, 1))
print('\n1) Người -1 vẽ cạnh thứ 4 của ô (0, 0):')
show_board(after)
print('   ô (0, 0) thuộc', after['boxes'][(0, 0)], '; người đi tiếp:', next_player(b, after, -1))
assert after['boxes'] == {(0, 0): -1} and next_player(b, after, -1) == -1
assert ('v', 0, 1) not in b['lines']      # result không sửa bàn cũ

# 2) Nước không đóng ô thì đổi lượt.
after2 = result(b, -1, ('h', 0, 1))
assert after2['boxes'] == {} and next_player(b, after2, -1) == 1
print('\n2) Nước không đóng ô: người đi tiếp là', next_player(b, after2, -1))

# 3) Một đoạn đóng cùng lúc 2 ô (đoạn chung của ô (0, 0) và ô (0, 1)).
b3 = new_board(2, 3)
for line in [('h', 0, 0), ('h', 1, 0), ('v', 0, 0), ('h', 0, 1), ('h', 1, 1), ('v', 0, 2)]:
    b3 = result(b3, 1, line)
b3_after = result(b3, -1, ('v', 0, 1))
print('\n3) Vẽ đoạn chung đóng 2 ô cùng lúc:')
show_board(b3_after)
assert b3_after['boxes'] == {(0, 0): -1, (0, 1): -1} and terminal(b3_after)
print('   utility theo người -1:', utility(b3_after, -1), '; theo người +1:', utility(b3_after, 1))
assert utility(b3_after, -1) == 1 and utility(b3_after, 1) == -1

# 4) Nước không hợp lệ bị từ chối: vẽ lại đoạn đã có, đoạn ra ngoài bàn.
for bad in [('h', 0, 0), ('h', 0, 2), ('v', 2, 0), ('x', 0, 0)]:
    try:
        result(b, 1, bad); print('LỖI: chấp nhận', bad)
    except ValueError:
        print('4) Từ chối nước không hợp lệ', bad)

# 5) Bàn khác kích thước: số đoạn đúng công thức rows·(cols−1) + (rows−1)·cols.
for rows, cols in [(2, 2), (3, 5), (5, 5)]:
    assert len(actions(new_board(rows, cols))) == count_lines(rows, cols)
print('5) Số nước đi ở bàn 2×2, 3×5, 5×5:', [len(actions(new_board(r, c))) for r, c in [(2, 2), (3, 5), (5, 5)]])

Bàn trống: 12 đoạn vẽ được; kết thúc? False

1) Người -1 vẽ cạnh thứ 4 của ô (0, 0):
+---+   +
| B |    
+---+   +
         
+   +   +
A (+1): 0 ô   B (-1): 1 ô   đã vẽ 4 đoạn
   ô (0, 0) thuộc -1 ; người đi tiếp: -1

2) Nước không đóng ô: người đi tiếp là 1

3) Vẽ đoạn chung đóng 2 ô cùng lúc:
+---+---+
| B | B |
+---+---+
A (+1): 0 ô   B (-1): 2 ô   đã vẽ 7 đoạn
   utility theo người -1: 1 ; theo người +1: -1
4) Từ chối nước không hợp lệ ('h', 0, 0)
4) Từ chối nước không hợp lệ ('h', 0, 2)
4) Từ chối nước không hợp lệ ('v', 2, 0)
4) Từ chối nước không hợp lệ ('x', 0, 0)
5) Số nước đi ở bàn 2×2, 3×5, 5×5: [4, 22, 40]


**[Bài 10] Nhận xét các hàm của Task 2.**
- Tất cả các hàm chỉ đọc `board['size']`, nên **chạy được với mọi kích thước bàn** (kiểm tra 5 ở trên: 4, 22 và 40 nước đi
  với bàn 2×2, 3×5 và 5×5 chấm, đúng công thức $L$).
- `result(board, player, action)` có cùng thứ tự tham số với `tictactoe.py` và **trả về bàn mới, không sửa bàn cũ**,
  để Alpha-Beta ở Task 3 thử nhiều nước trên cùng một bàn mà không phải hoàn tác.
- Sau mỗi nước chỉ cần xét **tối đa 2 ô kề đoạn vừa vẽ** (`adjacent_boxes`), không phải quét cả bàn.
- **Luật đi tiếp** nằm ở `next_player`: so số ô trước và sau nước đi, có ô mới thì người đó đi tiếp. Tách riêng hàm này vì
  cấu trúc dict chung của đề không có khóa "người đến lượt".
- Các kiểm tra ở trên cover đủ các tình huống quan trọng: đóng ô được đi tiếp (1), không đóng ô thì đổi lượt (2), một
  đoạn đóng **cùng lúc 2 ô** (3), từ chối nước vẽ lại và nước ra ngoài bàn (4).

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [6]:
# [Bài 10] Random agent: nhận bàn cờ (percept), trả một nước hợp lệ.
def random_player(board, player=None):
    """Chọn ngẫu nhiên đều một đoạn còn trống; `player` không dùng."""
    moves = actions(board)
    return moves[np.random.randint(len(moves))]

# Môi trường: hai agent lần lượt đi, đóng ô thì được đi tiếp.
def play_game(agent1, agent2, size=(4, 4), show_final_board=False):
    """Chơi 1 ván, agent1 là người +1 và đi trước. Trả utility theo góc nhìn +1."""
    board, player = new_board(*size), 1
    agents = {1: agent1, -1: agent2}
    while not terminal(board):
        action = agents[player](board, player)
        new = result(board, player, action)
        player = next_player(board, new, player)
        board = new
    if show_final_board:
        show_board(board)
    return utility(board, 1)

def play(agent1, agent2, N=1000, size=(4, 4)):
    """Cho hai agent đấu N ván; agent1 (+1) luôn đi trước."""
    results = {1: 0, -1: 0, 0: 0}
    for _ in range(N):
        results[play_game(agent1, agent2, size)] += 1
    return results

np.random.seed(42)
print('Một ván mẫu trên bàn 4×4 chấm, kết quả theo người +1:',
      play_game(random_player, random_player, show_final_board=True))

+---+---+---+
| A | A | A |
+---+---+---+
| B | B | A |
+---+---+---+
| B | B | A |
+---+---+---+
A (+1): 5 ô   B (-1): 4 ô   đã vẽ 24 đoạn
Một ván mẫu trên bàn 4×4 chấm, kết quả theo người +1: 1


Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [7]:
# [Bài 10] Hai random agent đấu 1000 ván trên nhiều kích thước bàn (seed 42).
import time
rows_out = []
for size in [(3, 3), (4, 4), (5, 5)]:
    np.random.seed(42)
    start = time.perf_counter()
    res = play(random_player, random_player, N=1000, size=size)
    elapsed = time.perf_counter() - start
    rows_out.append({'bàn (chấm)': f'{size[0]}×{size[1]}', 'số ô': count_boxes(*size),
                     'người +1 thắng (đi trước)': res[1], 'người -1 thắng': res[-1], 'hòa': res[0],
                     'tỷ lệ +1 thắng': res[1] / 1000, 'thời gian (s)': round(elapsed, 2)})
random_table = pd.DataFrame(rows_out)
random_table

,bàn (chấm),số ô,người +1 thắng (đi trước),người -1 thắng,hòa,tỷ lệ +1 thắng,thời gian (s)
0,3×3,4,400,432,168,0.400,0.07
1,4×4,9,490,510,0,0.490,0.17
2,5×5,16,451,466,83,0.451,0.35


**[Bài 10] Nhận xét: hai random agent đấu 1000 ván (người +1 luôn đi trước, seed 42).**
- **Bàn 4×4 chấm (mặc định của đề, 9 ô)**: người đi trước thắng **490**, người đi sau thắng **510**, **không có hòa**
  vì tổng số ô lẻ nên luôn có người hơn.
- Bàn 3×3 chấm (4 ô): **400 / 432 / 168 hòa**; bàn 5×5 chấm (16 ô): **451 / 466 / 83 hòa**. Hòa chỉ xảy ra ở bàn có số ô chẵn.
- **Kết quả này đúng như dự đoán.** Random agent không có chiến lược nên không khai thác được lợi thế nào; hai bên gần như
  chia đôi. Ở bàn 4×4, người đi trước thắng 490 ván, chỉ lệch 10 ván so với mức chia đều 500, trong khi độ lệch chuẩn của
  số ván thắng là khoảng $\sqrt{1000 \cdot 0{,}5 \cdot 0{,}5} \approx 16$ ván (khoảng 95%: 500 ± 31). Hai bàn còn lại cũng chỉ
  chênh 32 và 15 ván trên 1000. Vì vậy **không có lợi thế rõ ràng cho người đi trước** khi cả hai chơi ngẫu nhiên;
  chênh lệch quan sát được là dao động ngẫu nhiên.
- Thời gian 1000 ván chỉ dưới một giây ở cả ba kích thước, nên môi trường đủ nhanh cho các thí nghiệm ở Ngày 2 và 3.

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

### [Bài 10] Thiết kế Alpha-Beta có luật đi tiếp

Dựa trên `alpha_beta_search` của Tic-Tac-Toe (Bài 3), với hai thay đổi:

- **Luật đi tiếp.** Ở Tic-Tac-Toe, sau mỗi nước thì đổi lượt nên các tầng Max và Min xen kẽ nhau. Ở Dots and Boxes, người vừa
  đóng được ô được đi tiếp, nên sau mỗi nước, hàm `next_player` (Ngày 1) quyết định ai đi ở nút con. Nếu vẫn là mình thì nút con
  **vẫn là nút Max**; một nhánh có thể có nhiều tầng Max (hoặc Min) liên tiếp. Alpha và beta vẫn truyền xuống như cũ.
- **Giá trị.** Ở trạng thái kết thúc dùng `utility` của Task 1: +1 thắng, 0 hòa, −1 thua theo góc nhìn người tìm nước (`me`).
  Phần `cutoff` / `evaluate` chỉ dùng ở Task 4.

`alpha_beta_search` trả về nước đi, giá trị, **số nút** đã xét (biến `COUNT`) và thời gian; `time_limit` cho phép dừng khi tìm quá
lâu (khi đó `done = False`). `alpha_beta_player(board, player)` có cùng chữ ký với `random_player` nên đấu được với mọi agent.

Ô dưới thử trên hai bàn rất nhỏ: bàn 2×2 chấm (1 ô) có giá trị **−1** cho người đi trước, vì ô duy nhất luôn thuộc về người vẽ
đoạn thứ 4, tức người đi sau; bàn 2×3 chấm (2 ô) có giá trị **0** (hòa).

In [8]:
# [Bài 10] Task 3: Minimax với cắt tỉa Alpha-Beta, có luật đi tiếp.
import math
import time

class SearchTimeout(Exception):
    """Báo một lần tìm kiếm đã dùng hết thời gian cho phép."""

COUNT = 0          # số nút đã xét trong lần tìm kiếm gần nhất
DEADLINE = None    # thời điểm phải dừng; None = không giới hạn

def natural_order(board, player):
    """Không sắp xếp: thứ tự sinh nước của actions() (đoạn ngang trước, rồi đoạn dọc)."""
    return actions(board)

def ab_value(board, to_move, me, alpha, beta, order, depth=0, cutoff=None, evaluate=None):
    """Giá trị của `board` theo góc nhìn `me` khi `to_move` đến lượt."""
    global COUNT
    COUNT += 1
    if DEADLINE is not None and COUNT % 1000 == 0 and time.perf_counter() > DEADLINE:
        raise SearchTimeout
    if terminal(board):
        return utility(board, me)
    if cutoff is not None and depth >= cutoff:          # dùng ở Task 4
        return evaluate(board, me, to_move)
    maximizing = to_move == me
    value = -math.inf if maximizing else math.inf
    for action in order(board, to_move):
        child = result(board, to_move, action)
        # Luật đi tiếp: đóng được ô thì next_player trả lại chính to_move,
        # nên nút con vẫn là nút Max (hoặc vẫn là nút Min) thay vì đổi lượt như Tic-Tac-Toe.
        child_value = ab_value(child, next_player(board, child, to_move), me,
                               alpha, beta, order, depth + 1, cutoff, evaluate)
        if maximizing:
            value = max(value, child_value)
            if value >= beta:
                return value
            alpha = max(alpha, value)
        else:
            value = min(value, child_value)
            if value <= alpha:
                return value
            beta = min(beta, value)
    return value

def alpha_beta_search(board, player, order=natural_order, cutoff=None, evaluate=None, time_limit=None):
    """Tìm nước cho `player`. Trả nước đi, giá trị (+1 / 0 / -1, hoặc heuristic khi cắt), số nút, thời gian (s)
    và `done` (False nếu hết `time_limit` giây trước khi tìm xong)."""
    global COUNT, DEADLINE
    COUNT = 0
    start = time.perf_counter()
    DEADLINE = None if time_limit is None else start + time_limit
    best_move, best_value, alpha = None, -math.inf, -math.inf
    try:
        for action in order(board, player):
            child = result(board, player, action)
            value = ab_value(child, next_player(board, child, player), player,
                             alpha, math.inf, order, 1, cutoff, evaluate)
            if value > best_value:
                best_move, best_value = action, value
            alpha = max(alpha, best_value)
        done = True
    except SearchTimeout:
        best_move, best_value, done = None, None, False
    finally:
        DEADLINE = None
    return {'move': best_move, 'value': best_value, 'nodes': COUNT,
            'seconds': time.perf_counter() - start, 'done': done}

def alpha_beta_player(board, player=1):
    """Agent cùng chữ ký với random_player: nhận bàn cờ và người chơi, trả một nước hợp lệ."""
    return alpha_beta_search(board, player)['move']

for size in [(2, 2), (2, 3)]:
    r = alpha_beta_search(new_board(*size), 1)
    print(f"Bàn trống {size[0]}×{size[1]} chấm, người +1 đi trước: nước {r['move']}, giá trị {r['value']}, "
          f"{r['nodes']} nút, {1000 * r['seconds']:.1f} ms")

Bàn trống 2×2 chấm, người +1 đi trước: nước ('h', 0, 0), giá trị -1, 30 nút, 0.1 ms
Bàn trống 2×3 chấm, người +1 đi trước: nước ('v', 0, 1), giá trị 0, 1397 nút, 5.2 ms


Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [9]:
# [Bài 10] Thử Alpha-Beta trên các thế cờ tự tạo.
def make_board(rows, cols, lines, owners=None):
    """Tạo thế cờ từ danh sách đoạn đã vẽ và chủ của các ô đã đóng (kiểm tra cho khớp nhau)."""
    board = new_board(rows, cols)
    for line in lines:
        assert is_valid_line(board, line), line
        board['lines'][line] = True
    board['boxes'] = dict(owners or {})
    complete = {(r, c) for r in range(rows - 1) for c in range(cols - 1)
                if all(s in board['lines'] for s in box_sides((r, c)))}
    assert complete == set(board['boxes']), 'ô đã đủ 4 cạnh phải có chủ'
    return board

def move_values(board, player):
    """Giá trị chính xác (+1 / 0 / -1) của từng nước theo góc nhìn `player`: tìm tới hết ván cho mỗi nước."""
    values = {}
    for action in actions(board):
        child = result(board, player, action)
        values[action] = ab_value(child, next_player(board, child, player), player, -math.inf, math.inf, natural_order)
    return values

TESTS = {
    'lấy ô đang có 3 cạnh':  (make_board(3, 3, [('h', 0, 0), ('v', 0, 0), ('h', 1, 0)]), 1),
    'tránh vẽ cạnh thứ 3':   (make_board(3, 3, [('h', 0, 0), ('v', 0, 0), ('h', 1, 1), ('v', 1, 2)]), 1),
    'chuỗi 3 ô liên tiếp':   (make_board(2, 4, [('h', 0, 0), ('h', 0, 1), ('h', 0, 2), ('h', 1, 0), ('h', 1, 1),
                                                ('h', 1, 2), ('v', 0, 0)]), 1),
    'người -1 đến lượt':     (make_board(3, 3, [('h', 0, 0), ('h', 1, 0), ('v', 0, 0), ('v', 0, 1),
                                                ('h', 2, 1), ('v', 1, 2), ('v', 1, 1)], owners={(0, 0): 1}), -1),
}

rows = []
for name, (board, player) in TESTS.items():
    print(f'--- {name} (người {player:+d} đến lượt)')
    show_board(board)
    r = alpha_beta_search(board, player)
    values = move_values(board, player)
    best = max(values.values())
    rows.append({'thế cờ': name, 'người đi': player, 'nước chọn': r['move'], 'giá trị': r['value'],
                 'đóng ô?': closed_box(board, result(board, player, r['move'])),
                 'nước tối ưu (giá trị tốt nhất)': sorted(a for a, v in values.items() if v == best),
                 'nước tối ưu?': values[r['move']] == best,
                 'số nút': r['nodes'], 'thời gian (ms)': round(1000 * r['seconds'], 2)})
test_table = pd.DataFrame(rows)
test_table

--- lấy ô đang có 3 cạnh (người +1 đến lượt)
+---+   +
|        
+---+   +
         
+   +   +
A (+1): 0 ô   B (-1): 0 ô   đã vẽ 3 đoạn
--- tránh vẽ cạnh thứ 3 (người +1 đến lượt)
+---+   +
|        
+   +---+
        |
+   +   +
A (+1): 0 ô   B (-1): 0 ô   đã vẽ 4 đoạn
--- chuỗi 3 ô liên tiếp (người +1 đến lượt)
+---+---+---+
|            
+---+---+---+
A (+1): 0 ô   B (-1): 0 ô   đã vẽ 7 đoạn
--- người -1 đến lượt (người -1 đến lượt)
+---+   +
| A |    
+---+   +
    |   |
+   +---+
A (+1): 1 ô   B (-1): 0 ô   đã vẽ 7 đoạn


,thế cờ,người đi,nước chọn,giá trị,đóng ô?,nước tối ưu (giá trị tốt nhất),nước tối ưu?,số nút,thời gian (ms)
0,lấy ô đang có 3 cạnh,1,"(v, 0, 1)",0,True,"[(v, 0, 1)]",True,18109,76.65
1,tránh vẽ cạnh thứ 3,1,"(h, 0, 1)",1,False,"[(h, 0, 1), (h, 2, 0), (v, 0, 2), (v, 1, 0)]",True,3034,12.75
2,chuỗi 3 ô liên tiếp,1,"(v, 0, 1)",1,True,"[(v, 0, 1)]",True,11,0.07
3,người -1 đến lượt,-1,"(h, 1, 1)",0,True,"[(h, 1, 1)]",True,122,0.52


**[Bài 10] Nhận xét: agent có nhận ra cơ hội không?** Cột "nước tối ưu (giá trị tốt nhất)" được tính độc lập: thử từng nước rồi
tìm tới hết ván. Agent chọn **nước tối ưu ở cả 4 thế**:
- **Lấy ô đang có 3 cạnh:** ô (0, 0) thiếu một cạnh; agent vẽ `('v', 0, 1)` để đóng ô và đây là **nước tối ưu duy nhất**. Giá trị
  là 0: lấy ô này giữ được hòa, còn bỏ qua thì đối thủ lấy và người +1 thua.
- **Tránh vẽ cạnh thứ 3:** chưa ô nào lấy được ngay. Agent chọn `('h', 0, 1)`, một nước **an toàn**: ô kề nó chỉ lên 2 cạnh, không
  tặng ô cho đối thủ. Giá trị +1: với bàn này người +1 thắng được nếu chơi đúng. Cả 4 nước tối ưu đều là nước an toàn.
- **Chuỗi 3 ô liên tiếp:** ô đầu chuỗi có 3 cạnh, hai ô sau có 2 cạnh. Agent vẽ `('v', 0, 1)`, đóng ô đầu chuỗi, giá trị +1.
  Chỉ cần **11 nút** vì sau mỗi ô đóng được, agent lại được đi và lấy tiếp (xem ô dưới).
- **Người −1 đến lượt:** người +1 đã có ô (0, 0); ô (1, 1) có 3 cạnh. Agent (cầm −1) lấy ngay ô này bằng `('h', 1, 1)`, đưa ván
  về hòa (giá trị 0). Kết quả này cũng xác nhận agent chơi đúng khi cầm người −1.

In [10]:
# [Bài 10] Đi tiếp thế "chuỗi 3 ô" tới hết ván, hai bên đều là Alpha-Beta, để thấy luật đi tiếp.
def play_from(board, player, agents):
    """Chơi tiếp từ một thế cờ, in từng nước và ai đi tiếp."""
    while not terminal(board):
        action = agents[player](board, player)
        child = result(board, player, action)
        nxt = next_player(board, child, player)
        print(f'người {player:+d} vẽ {action}' + ('  → đóng ô, đi tiếp' if nxt == player else ''))
        board, player = child, nxt
    show_board(board)
    return utility(board, 1)

play_from(TESTS['chuỗi 3 ô liên tiếp'][0], 1, {1: alpha_beta_player, -1: alpha_beta_player})

người +1 vẽ ('v', 0, 1)  → đóng ô, đi tiếp
người +1 vẽ ('v', 0, 2)  → đóng ô, đi tiếp
người +1 vẽ ('v', 0, 3)  → đóng ô, đi tiếp
+---+---+---+
| A | A | A |
+---+---+---+
A (+1): 3 ô   B (-1): 0 ô   đã vẽ 10 đoạn


1

**[Bài 10] Nhận xét luật đi tiếp.** Người +1 vẽ 3 nước liền nhau: mỗi nước đóng một ô nên được đi tiếp, lấy trọn chuỗi
3 ô và thắng 3–0, người −1 không được đi nước nào. Đây chính là luật mà tìm kiếm phải mô hình hóa: một lượt có thể gồm nhiều nước.

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [11]:
# [Bài 10] Thời gian một nước trên bàn trống khi tăng kích thước (Alpha-Beta không sắp xếp, giới hạn 20 giây).
LIMIT = 20
rows = []
for size in [(2, 2), (2, 3), (2, 4), (3, 3), (2, 5)]:
    r = alpha_beta_search(new_board(*size), 1, time_limit=LIMIT)
    rows.append({'bàn (chấm)': f'{size[0]}×{size[1]}', 'số ô': count_boxes(*size), 'số cạnh L': count_lines(*size),
                 'giải xong?': r['done'], 'nước chọn': r['move'], 'giá trị': r['value'],
                 'số nút': r['nodes'], 'thời gian (s)': round(r['seconds'], 3)})
timing_table = pd.DataFrame(rows)
timing_table

,bàn (chấm),số ô,số cạnh L,giải xong?,nước chọn,giá trị,số nút,thời gian (s)
0,2×2,1,4,True,"(h, 0, 0)",-1.0,30,0.000
1,2×3,2,7,True,"(v, 0, 1)",0.0,1397,0.005
2,2×4,3,10,True,"(h, 0, 0)",-1.0,32096,0.128
3,3×3,4,12,True,"(h, 0, 0)",1.0,440721,1.904
4,2×5,4,13,False,None,NaN,4592000,20.001


**[Bài 10] Nhận xét: thời gian một nước và bàn lớn nhất giải được.**
- Bàn trống là trường hợp khó nhất vì phải tìm tới hết ván. Số nút tăng rất nhanh theo số cạnh $L$: **30, 1.397, 32.096 và
  440.721 nút** cho bàn 2×2, 2×3, 2×4, 3×3 chấm ($L$ = 4, 7, 10, 12), tức gấp khoảng 14–47 lần mỗi khi thêm vài cạnh.
- Bàn **3×3 chấm (4 ô)** là bàn lớn nhất giải được trong vài giây: người đi trước **thắng** (giá trị +1).
- Bàn **2×5 chấm** cũng chỉ có 4 ô nhưng nhiều hơn một cạnh ($L$ = 13), và **không giải xong trong 20 giây**. Bàn chuẩn 5×5
  chấm có $L$ = 40 nên hoàn toàn không thể tìm tới hết ván bằng cách này.
- Thời gian cụ thể (cột cuối) phụ thuộc máy; số nút của các lần tìm xong thì cố định.

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

### [Bài 10] Move ordering

Alpha-Beta cắt tỉa được nhiều hơn khi gặp nước tốt sớm. Trong Dots and Boxes có thể đoán nước tốt khá chắc chỉ bằng cách xem các
ô kề đoạn sẽ vẽ (hàm `move_kind`):
- **Đóng ô** (ô kề đang có 3 cạnh): được thêm điểm và được đi tiếp, thường là nước tốt nhất.
- **An toàn** (không ô kề nào lên 3 cạnh): không tặng gì cho đối thủ.
- **Tặng ô** (tạo cạnh thứ 3 cho một ô): đối thủ lấy được ngay, thường là nước tệ.

So sánh 3 cách: không sắp xếp; chỉ đưa nước đóng ô lên trước; và đầy đủ "đóng ô → an toàn → tặng ô". Giá trị Minimax phải giống
nhau, chỉ khác số nút và thời gian.

In [12]:
# [Bài 10] Move ordering: so sánh 3 cách sắp xếp nước đi (cùng thuật toán, giới hạn 60 giây mỗi lần tìm).
def box_side_count(board, box):
    return sum(side in board['lines'] for side in box_sides(box))

def move_kind(board, action):
    """0 = đóng được ô; 1 = an toàn (không ô kề nào lên 3 cạnh); 2 = tặng ô (tạo cạnh thứ 3 cho một ô)."""
    counts = [box_side_count(board, box) for box in adjacent_boxes(board, action)]
    if 3 in counts:
        return 0
    return 2 if 2 in counts else 1

def capture_first_order(board, player):
    """Nước đóng ô lên trước, còn lại giữ thứ tự cũ."""
    return sorted(actions(board), key=lambda a: move_kind(board, a) != 0)

def smart_order(board, player):
    """Đóng ô → an toàn → tặng ô."""
    return sorted(actions(board), key=lambda a: move_kind(board, a))

ORDERS = {'không sắp xếp': natural_order, 'đóng ô trước': capture_first_order,
          'đóng ô → an toàn → tặng ô': smart_order}
ORDER_BOARDS = {f'bàn trống {r}×{c}': (new_board(r, c), 1) for r, c in [(2, 4), (3, 3), (2, 5)]}
ORDER_BOARDS.update(TESTS)

ORDER_LIMIT = 60
rows = []
for name, (board, player) in ORDER_BOARDS.items():
    for label, order in ORDERS.items():
        if name == 'bàn trống 2×5' and label == 'không sắp xếp':
            # Đã chạy ở bảng thời gian phía trên và không xong trong 20 giây; không chạy lại.
            t = timing_table[timing_table['bàn (chấm)'] == '2×5'].iloc[0]
            r = {'done': False, 'value': None, 'nodes': t['số nút'], 'seconds': t['thời gian (s)']}
        else:
            r = alpha_beta_search(board, player, order=order, time_limit=ORDER_LIMIT)
        rows.append({'thế cờ': name, 'cách sắp xếp': label, 'giải xong?': r['done'], 'giá trị': r['value'],
                     'số nút': r['nodes'], 'thời gian (s)': round(r['seconds'], 3)})
ordering_table = pd.DataFrame(rows)
ordering_table.pivot(index='thế cờ', columns='cách sắp xếp', values=['số nút', 'thời gian (s)']) \
              .reindex(columns=list(ORDERS), level=1).reindex(list(ORDER_BOARDS))

số nút                                         \
cách sắp xếp         không sắp xếp đóng ô trước đóng ô → an toàn → tặng ô   
thế cờ                                                                      
bàn trống 2×4              32096.0      12619.0                   11131.0   
bàn trống 3×3             440721.0     111946.0                   91032.0   
bàn trống 2×5            4592000.0     772453.0                  588158.0   
lấy ô đang có 3 cạnh       18109.0       3133.0                    2145.0   
tránh vẽ cạnh thứ 3         3034.0        842.0                     652.0   
chuỗi 3 ô liên tiếp           11.0         11.0                      11.0   
người -1 đến lượt            122.0         45.0                      45.0   

                     thời gian (s)                                         
cách sắp xếp         không sắp xếp đóng ô trước đóng ô → an toàn → tặng ô  
thế cờ                                                                     
bàn trống 2×4                0.126        0.090                     0.080  
bàn trống 3×3                1.897        0.862                     0.715  
bàn trống 2×5               20.001        6.149                     4.745  
lấy ô đang có 3 cạnh         0.078        0.023                     0.016  
tránh vẽ cạnh thứ 3          0.013        0.006                     0.005  
chuỗi 3 ô liên tiếp          0.000        0.000                     0.000  
người -1 đến lượt            0.001        0.000                     0.000

**[Bài 10] Nhận xét move ordering.**
- **Giá trị không đổi** với cả 3 cách (cột giá trị trong `ordering_table` giống nhau ở mọi thế đã giải xong), đúng lý thuyết:
  sắp xếp chỉ đổi thứ tự duyệt.
- **Số nút giảm mạnh:** bàn trống 3×3 từ 440.721 xuống 111.946 ("đóng ô trước") và **91.032** (đầy đủ), giảm khoảng **4,8 lần**;
  bàn 2×4 giảm khoảng 2,9 lần; thế "lấy ô đang có 3 cạnh" giảm khoảng **8 lần** (18.109 → 2.145).
- **Bàn 2×5 giải được nhờ sắp xếp:** không sắp xếp thì không xong trong 20 giây; có sắp xếp thì xong với 772.453 nút ("đóng ô
  trước") và 588.158 nút (đầy đủ), mất vài giây.
- Phần lớn lợi ích đến từ việc **đưa nước đóng ô lên trước**; thêm bước "an toàn trước tặng ô" giảm thêm khoảng 12–32% số nút
  ở hầu hết các thế (không đổi ở thế "người −1 đến lượt").
- **Thời gian giảm ít hơn số nút** (bàn 3×3: khoảng 2–3 lần so với 4,8 lần), vì mỗi nút phải đếm cạnh của các ô kề để sắp xếp.
- Thế "chuỗi 3 ô" không đổi (11 nút) vì chỉ có đúng một nước đóng ô ở mỗi bước.

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

In [13]:
# [Bài 10] Nước đầu tiên: ghi nhớ trạng thái đã giải (transposition table).
# Nhận xét then chốt: số ô hai bên còn giành thêm được từ một thế chỉ phụ thuộc vào TẬP CẠNH ĐÃ VẼ,
# không phụ thuộc thứ tự vẽ hay ô nào thuộc ai. Mỗi tập cạnh chỉ cần giải một lần: tối đa 2^L trạng thái thay vì L! ván.
from functools import lru_cache

def solve_margin(rows, cols):
    """Hiệu số ô tốt nhất mà người đi trước đạt được khi hai bên chơi tối ưu (Minimax có ghi nhớ),
    cùng số trạng thái đã giải và thời gian. Tập cạnh được mã hóa thành một số nguyên (mỗi cạnh một bit)."""
    lines = all_lines(new_board(rows, cols))
    bit = {line: i for i, line in enumerate(lines)}
    boxes = [(r, c) for r in range(rows - 1) for c in range(cols - 1)]
    box_mask = [sum(1 << bit[s] for s in box_sides(b)) for b in boxes]
    touching = [[k for k, m in enumerate(box_mask) if m >> i & 1] for i in range(len(lines))]
    full = (1 << len(lines)) - 1

    @lru_cache(maxsize=None)
    def best(drawn):
        """Hiệu số ô (của mình trừ đối thủ) tối đa còn giành được khi đến lượt mình ở tập cạnh `drawn`."""
        if drawn == full:
            return 0
        value = -math.inf
        for i in range(len(lines)):
            if drawn >> i & 1:
                continue
            new = drawn | (1 << i)
            closed = sum(1 for k in touching[i] if new & box_mask[k] == box_mask[k])
            # đóng được ô: cộng số ô và đi tiếp; không đóng: đổi lượt nên lấy âm giá trị của đối thủ
            value = max(value, closed + best(new) if closed else -best(new))
        return value

    start = time.perf_counter()
    margin = best(0)
    return {'margin': margin, 'states': best.cache_info().currsize, 'seconds': time.perf_counter() - start}

# Đối chiếu với Alpha-Beta có sắp xếp nước ở bảng move ordering (bàn 3×4 không chạy Alpha-Beta vì quá lâu).
smart_rows = ordering_table[ordering_table['cách sắp xếp'] == 'đóng ô → an toàn → tặng ô'].set_index('thế cờ')
rows = []
for size in [(2, 4), (3, 3), (2, 5), (3, 4)]:
    s = solve_margin(*size)
    key = f'bàn trống {size[0]}×{size[1]}'
    ab = smart_rows.loc[key] if key in smart_rows.index else None
    rows.append({'bàn (chấm)': f'{size[0]}×{size[1]}', 'số cạnh L': count_lines(*size),
                 'hiệu số ô của người đi trước': s['margin'], 'kết quả': 'thắng' if s['margin'] > 0 else
                 ('thua' if s['margin'] < 0 else 'hòa'),
                 'trạng thái đã giải': s['states'], 'có ghi nhớ (s)': round(s['seconds'], 3),
                 'Alpha-Beta sắp xếp (s)': ab['thời gian (s)'] if ab is not None else '–',
                 'giá trị Alpha-Beta': ab['giá trị'] if ab is not None else '–'})
memo_table = pd.DataFrame(rows)
display(memo_table)

# Ngoại suy cho bàn lớn hơn: số trạng thái tối đa 2^L, thời gian theo tốc độ đo được trên bàn 3×4.
per_state = memo_table.iloc[-1]['có ghi nhớ (s)'] / memo_table.iloc[-1]['trạng thái đã giải']
pd.DataFrame([{'bàn (chấm)': f'{r}×{c}', 'số cạnh L': count_lines(r, c), 'trạng thái 2^L': f'{2 ** count_lines(r, c):.2e}',
               'thời gian ước lượng': f'{per_state * 2 ** count_lines(r, c):.2e} s'} for r, c in [(4, 4), (4, 5), (5, 5)]])

,bàn (chấm),số cạnh L,hiệu số ô của người đi trước,kết quả,trạng thái đã giải,có ghi nhớ (s),Alpha-Beta sắp xếp (s),giá trị Alpha-Beta
0,2×4,10,-1,thua,1024,0.003,0.08,-1.0
1,3×3,12,2,thắng,4096,0.013,0.715,1.0
2,2×5,13,0,hòa,8192,0.027,4.745,0.0
3,3×4,17,-2,thua,131072,0.577,–,–


,bàn (chấm),số cạnh L,trạng thái 2^L,thời gian ước lượng
0,4×4,24,1.68e+07,7.39e+01 s
1,4×5,31,2.15e+09,9.45e+03 s
2,5×5,40,1.10e+12,4.84e+06 s


**[Bài 10] Nhận xét: làm gì với các nước đầu tiên?** Bàn trống là trường hợp xấu nhất. Ô trên thử cách **ghi nhớ trạng thái đã
giải** (transposition table):
- Số ô hai bên còn giành thêm được chỉ phụ thuộc vào **tập cạnh đã vẽ**, không phụ thuộc thứ tự vẽ. Mỗi tập cạnh chỉ cần giải một
  lần: tối đa $2^L$ trạng thái, thay vì $L!$ thứ tự vẽ của cây trò chơi (xem Task 1). Số trạng thái đã giải đúng bằng $2^L$
  (1.024; 4.096; 8.192; 131.072).
- Kết quả **khớp với Alpha-Beta**: bàn 2×4 người đi trước thua 1 ô; bàn 3×3 thắng 2 ô; bàn 2×5 hòa. Ghi nhớ còn giải được bàn
  **3×4 chấm** (6 ô): người đi trước **thua 2 ô**, trong chưa tới một giây, trong khi Alpha-Beta (có sắp xếp) trên bàn 2×5 nhỏ hơn đã mất vài giây.
- Ngoại suy theo tốc độ đo được: bàn **4×4** chấm cần khoảng $1{,}7 \times 10^7$ trạng thái (khoảng hơn một phút, nhưng cần vài GB bộ nhớ
  trong Python), bàn 4×5 khoảng vài giờ, bàn chuẩn **5×5** khoảng $10^{12}$ trạng thái, cỡ hàng tháng. Vì vậy với bàn chuẩn:
  - dùng **cutoff + heuristic** (Task 4) để chỉ nhìn trước vài nước;
  - **sách khai cuộc**: giải trước các nước đầu một lần rồi lưu lại;
  - giảm trạng thái bằng **đối xứng** của bàn (xoay, lật; bàn vuông có tới 8 phép đối xứng);
  - ở các nước đầu, phần lớn nước đều an toàn và gần như tương đương, nên có thể chọn nhanh một nước an toàn và chỉ tìm sâu khi
    bàn đã có nhiều ô 2 cạnh.

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [14]:
# [Bài 10] Alpha-Beta (sắp xếp nước "đóng ô → an toàn → tặng ô") đấu Random trên bàn nhỏ 2×4 chấm (3 ô), 100 ván mỗi vai.
def measured(agent, times):
    def wrapped(board, player=1):
        start = time.perf_counter()
        move = agent(board, player)
        times.append(time.perf_counter() - start)
        return move
    return wrapped

def ab_smart_player(board, player=1):
    return alpha_beta_search(board, player, order=smart_order)['move']

rows = []
for role in (1, -1):
    np.random.seed(42)
    times = []
    agent = measured(ab_smart_player, times)
    res = play(agent, random_player, N=100, size=(2, 4)) if role == 1 else play(random_player, agent, N=100, size=(2, 4))
    rows.append({'Alpha-Beta là': f'người {role:+d}' + (' (đi trước)' if role == 1 else ' (đi sau)'),
                 'thắng': res[role], 'hòa': res[0], 'thua': res[-role],
                 'TG TB/nước (ms)': round(1000 * float(np.mean(times)), 2)})
pd.DataFrame(rows)

,Alpha-Beta là,thắng,hòa,thua,TG TB/nước (ms)
0,người +1 (đi trước),77,0,23,15.43
1,người -1 (đi sau),100,0,0,4.17


**[Bài 10] Nhận xét Alpha-Beta đấu Random (bàn 2×4 chấm, 3 ô, 100 ván mỗi vai, seed 42).**
- **Cầm người −1 (đi sau): thắng cả 100 ván.** Theo bảng ở trên, bàn 2×4 là thế thắng của người đi sau, và agent chơi đúng nên
  không để mất ván nào.
- **Cầm người +1 (đi trước): thắng 77, thua 23.** Bàn 2×4 là thế **thua** của người đi trước (giá trị −1). Khi mọi nước đều có giá
  trị −1, agent coi chúng như nhau và lấy nước đầu tiên, không cố đi nước "gây khó" cho đối thủ. Random thường đi sai nên agent
  vẫn thắng phần lớn, nhưng khi Random tình cờ đi đúng thì agent thua.
- Không có ván hòa vì bàn có 3 ô (số lẻ).
- Đây là hạn chế của utility chỉ có thắng/hòa/thua. Có thể cải thiện bằng cách dùng **hiệu số ô** làm utility, hoặc khi mọi nước
  cùng thua thì ưu tiên nước khiến đối thủ phải tìm đúng nhiều nước nhất.
- Thời gian mỗi nước khi đi trước lâu hơn vì nước đầu phải tìm trên bàn trống (số cụ thể ở bảng, phụ thuộc máy).

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

### [Bài 10] Hàm heuristic

$$h(s) = \frac{\text{(số ô của mình} - \text{số ô của đối thủ)} \pm \text{(số ô đang có 3 cạnh)}}{\text{tổng số ô} + 1}$$

- **Hiệu số ô hiện tại** là phần chắc chắn: ô đã đóng không đổi chủ.
- **Ô đang có 3 cạnh** lấy được ngay bởi người đến lượt, nên được **cộng** nếu mình đến lượt và **trừ** nếu đối thủ đến lượt.
- Chia cho (tổng số ô + 1) để $h$ luôn nằm trong $(-1, 1)$, nhỏ hơn giá trị thắng/thua thật (±1). Nhờ vậy agent không bao giờ
  coi một thế chưa kết thúc tốt hơn một nước thắng thật.
- Hạn chế: chưa tính **chuỗi** ô (một ô 3 cạnh có thể kéo theo cả chuỗi ô đi tiếp) và chưa biết ai sẽ buộc phải tặng ô ở cuối ván.

In [15]:
# [Bài 10] Task 4: hàm đánh giá heuristic.
def heuristic(board, me, to_move):
    """Ước lượng một thế chưa kết thúc theo góc nhìn `me`, luôn nằm trong (-1, 1) để không lẫn với thắng/thua thật.
    = (hiệu số ô hiện tại ± số ô đang có 3 cạnh) / (số ô + 1):
    ô có 3 cạnh sẽ thuộc về người đến lượt (lấy ngay được), nên cộng nếu `me` đến lượt, trừ nếu đối thủ đến lượt."""
    rows, cols = board['size']
    n_boxes = (rows - 1) * (cols - 1)
    ready = sum(1 for r in range(rows - 1) for c in range(cols - 1)
                if (r, c) not in board['boxes'] and box_side_count(board, (r, c)) == 3)
    margin = score(board, me) + (ready if to_move == me else -ready)
    return margin / (n_boxes + 1)

rows = []
for name, (board, player) in TESTS.items():
    rows.append({'thế cờ': name, 'người đến lượt': player, 'h theo người +1': round(heuristic(board, 1, player), 3),
                 'h theo người -1': round(heuristic(board, -1, player), 3)})
rows.append({'thế cờ': 'bàn trống 4×4', 'người đến lượt': 1,
             'h theo người +1': heuristic(new_board(4, 4), 1, 1), 'h theo người -1': heuristic(new_board(4, 4), -1, 1)})
pd.DataFrame(rows)

,thế cờ,người đến lượt,h theo người +1,h theo người -1
0,lấy ô đang có 3 cạnh,1,0.20,-0.20
1,tránh vẽ cạnh thứ 3,1,0.00,0.00
2,chuỗi 3 ô liên tiếp,1,0.25,-0.25
3,người -1 đến lượt,-1,0.00,0.00
4,bàn trống 4×4,1,0.00,0.00


**[Bài 10] Nhận xét heuristic.** Ở thế "lấy ô đang có 3 cạnh", người +1 đến lượt và lấy được 1 ô nên $h = 1/5 = 0{,}2$; ở thế
"chuỗi 3 ô" là $1/4 = 0{,}25$ (bàn chỉ có 3 ô). Ở thế "người −1 đến lượt", người +1 đang hơn 1 ô nhưng người −1 sắp lấy lại 1 ô nên
$h = 0$. Giá trị theo hai người luôn đối nhau (đối xứng). Bàn trống và các thế chưa có ô nào 3 cạnh đều cho 0: heuristic không phân
biệt được các nước đầu ván.

### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [16]:
# [Bài 10] Cắt tìm kiếm ở độ sâu cutoff và dùng heuristic (cùng hàm alpha_beta_search, thứ tự "đóng ô → an toàn → tặng ô").
# (a) Các thế cờ tự tạo: so nước chọn với nước tối ưu tìm tới hết ván.
rows = []
for name, (board, player) in TESTS.items():
    values = move_values(board, player)
    best = max(values.values())
    for cutoff in (1, 2, 4, None):
        r = alpha_beta_search(board, player, order=smart_order, cutoff=cutoff, evaluate=heuristic)
        rows.append({'thế cờ': name, 'cutoff': 'không' if cutoff is None else cutoff, 'nước chọn': r['move'],
                     'giá trị (ước lượng)': round(r['value'], 3), 'nước tối ưu?': values[r['move']] == best,
                     'số nút': r['nodes'], 'thời gian (ms)': round(1000 * r['seconds'], 2)})
display(pd.DataFrame(rows))

# (b) Bàn chuẩn 4×4 và 5×5 chấm (không thể tìm tới hết ván): số nút và thời gian một nước theo cutoff.
rows = []
for size in [(4, 4), (5, 5)]:
    for cutoff in (2, 4, 6):
        r = alpha_beta_search(new_board(*size), 1, order=smart_order, cutoff=cutoff, evaluate=heuristic, time_limit=60)
        rows.append({'bàn (chấm)': f'{size[0]}×{size[1]}', 'cutoff': cutoff, 'xong?': r['done'], 'nước chọn': r['move'],
                     'số nút': r['nodes'], 'thời gian (s)': round(r['seconds'], 3)})
pd.DataFrame(rows)

,thế cờ,cutoff,nước chọn,giá trị (ước lượng),nước tối ưu?,số nút,thời gian (ms)
0,lấy ô đang có 3 cạnh,1,"(v, 0, 1)",0.20,True,9,0.09
1,lấy ô đang có 3 cạnh,2,"(v, 0, 1)",0.20,True,25,0.28
2,lấy ô đang có 3 cạnh,4,"(v, 0, 1)",0.20,True,149,1.28
3,lấy ô đang có 3 cạnh,không,"(v, 0, 1)",0.00,True,2145,16.51
4,tránh vẽ cạnh thứ 3,1,"(h, 0, 1)",0.00,True,8,0.08
5,tránh vẽ cạnh thứ 3,2,"(h, 0, 1)",0.00,True,22,0.24
6,tránh vẽ cạnh thứ 3,4,"(h, 0, 1)",0.20,True,116,1.12
7,tránh vẽ cạnh thứ 3,không,"(h, 0, 1)",1.00,True,652,5.07
8,chuỗi 3 ô liên tiếp,1,"(v, 0, 1)",0.50,True,3,0.03
9,chuỗi 3 ô liên tiếp,2,"(v, 0, 1)",0.75,True,7,0.05


,bàn (chấm),cutoff,xong?,nước chọn,số nút,thời gian (s)
0,4×4,2,True,"(h, 0, 0)",70,0.002
1,4×4,4,True,"(h, 0, 0)",1630,0.036
2,4×4,6,True,"(h, 0, 0)",32072,0.678
3,5×5,2,True,"(h, 0, 0)",118,0.004
4,5×5,4,True,"(h, 0, 0)",4638,0.162
5,5×5,6,True,"(h, 0, 0)",165400,5.645


**[Bài 10] Nhận xét cutoff.**
- **Trên 4 thế tự tạo, mọi mức cutoff (1, 2, 4) đều chọn nước tối ưu**, giống tìm tới hết ván, nhưng ít nút hơn nhiều: ví dụ thế
  "lấy ô đang có 3 cạnh" cần 9 / 25 / 149 nút thay vì 2.145; thế "tránh vẽ cạnh thứ 3" cần 8 / 22 / 116 thay vì 652.
- **Giá trị ước lượng khác giá trị thật** khi cutoff nhỏ: thế "tránh vẽ cạnh thứ 3" thật ra thắng được (+1), nhưng cutoff 1–4 chỉ
  thấy 0 hoặc 0,2 vì chưa nhìn tới lúc phân thắng thua. Nước chọn vẫn đúng vì heuristic xếp hạng các nước đúng thứ tự.
- **Bàn chuẩn:** trên bàn 4×4 chấm, cutoff 2 / 4 / 6 xét **70 / 1.630 / 32.072 nút**; trên bàn 5×5 chấm là **118 / 4.638 / 165.400
  nút**. Số nút tăng khoảng 20–40 lần mỗi khi tăng cutoff thêm 2. Cutoff 4 chỉ mất một phần nhỏ của giây nên dùng được để chơi;
  cutoff 6 trên bàn 5×5 đã mất vài giây mỗi nước.
- Trên bàn trống, mọi mức cutoff đều chọn `('h', 0, 0)`: các thế sau vài nước đầu đều có $h = 0$, nên agent lấy nước đầu tiên theo thứ
  tự. Đây là giới hạn của heuristic đã nêu ở trên; một heuristic có tính chuỗi ô sẽ phân biệt tốt hơn (Ngày 3).

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [17]:
# Your code/ answer goes here.

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [18]:
# Your code/ answer goes here.

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

In [19]:
# Your code/ answer goes here.

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [20]:
# Your code/ answer goes here.